[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/05_strings.ipynb)

*Python from Scratch · Notebook 5 of 12*

# Strings

Text in Python: indexing, slicing, string methods, formatting and why strings can't be changed in place.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A string is a row of letters you can read but never overwrite. Every 'change' builds a new string.

**What interviewers really test here:** Immutability, slicing, join vs + in a loop, and counting characters with a dictionary.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Creating strings *(Basics)*
- [ ] 2. Indexing and slicing *(Basics)*
- [ ] 3. Strings are immutable *(Core)*
- [ ] 4. The most useful string methods *(Core)*
- [ ] 5. Working character by character *(Core)*
- [ ] 6. Formatting text *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **string** is an **immutable sequence of Unicode characters**. In Python its type is `str`. There is no separate character type — a single character is a string of length 1.



As a **sequence**, a string supports indexing, slicing, `len()`, iteration, membership (`in`), concatenation (`+`) and repetition (`*`).

### Creating strings and escape sequences

| Literal | Use |
| --- | --- |
| `'text'`, `"text"` | identical; pick the one that avoids escaping quotes inside |
| `'''text'''`, `"""text"""` | multi-line strings, docstrings |
| `r"C:\path"` | raw string — backslashes are literal (regex, Windows paths) |
| `f"Hi {name}"` | formatted string literal (f-string) |
| `b"bytes"` | bytes, not text (binary data) |



| Escape | Meaning |
| --- | --- |
| `\n` | new line |
| `\t` | tab |
| `\\` | backslash |
| `\'` `\"` | quote |
| `\u20B9` | Unicode character (₹) |

### Indexing and slicing rules

- Positive indices run 0 … len − 1 from the left; negative indices run −1 … −len from the right.
- `s[i]` outside the range raises `IndexError`.
- `s[start:stop:step]` returns a new string with characters from start up to (not including) stop.
- Missing start/stop default to the ends; missing step defaults to 1.
- Out-of-range slice bounds are clipped, never an error.
- A negative step walks backwards; then the defaults are reversed: `s[::-1]` is the reverse.
- `s[:k] + s[k:] == s` for any k.

### Immutability

Once created, a string object can never change. Assignment to an index (`s[0] = 'x'`) raises `TypeError`, and methods like `upper()` or `replace()` return **new** strings.



#### Why immutable?




- Strings can be **dictionary keys** and set members (their hash never changes).
- They are safe to share between variables and threads.
- Python can reuse identical small strings (**interning**).



#### Consequence



Repeated `s += piece` in a loop may copy the string every time — O(n²) in total. The idiom is to collect pieces in a list and call `"".join(pieces)` once — O(n).

### Methods by category

| Category | Methods |
| --- | --- |
| Case | `lower upper title capitalize swapcase casefold` |
| Whitespace | `strip lstrip rstrip` |
| Search | `find rfind index rindex count startswith endswith`, operator `in` |
| Replace / split / join | `replace split rsplit splitlines partition join` |
| Tests (return bool) | `isdigit isnumeric isalpha isalnum isspace isupper islower istitle` |
| Alignment | `center ljust rjust zfill` |
| Encoding | `encode` (str → bytes); `bytes.decode` |



`find` returns −1 when not found; `index` raises `ValueError`. Built-ins that work on strings: `len, ord, chr, str, sorted, reversed, min, max`.

### String comparison and Unicode

Strings compare **lexicographically** by Unicode code points, character by character: `"apple" < "banana"`, `"Zebra" < "apple"` (uppercase codes 65–90 come before lowercase 97–122), and `"app" < "apple"` (a prefix is smaller).



For case-insensitive comparison use `a.casefold() == b.casefold()`. Python 3 strings are Unicode, so they hold any language: `len("नमस्ते")` counts code points.

---
# Part 2: Lessons, step by step

## Lesson 1: Creating strings

*Level: Basics*

A **string** (`str`) is a sequence of characters. You can write one with single, double or triple quotes:



```python
a = 'single'
b = "double — can contain 'single' quotes"
c = """triple quotes
span several lines"""
```



**Escape sequences** put special characters inside a string: `\n` new line, `\t` tab, `\\` a backslash, `\"` a quote. A **raw string** `r"C:\new"` keeps backslashes as they are.



`len(s)` gives the number of characters. `+` joins strings and `*` repeats them.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def creating_demo():
    first = "Py"
    second = 'thon'
    word = first + second
    line = "-" * 10
    size = len(word)
    print(word, size)
    print(line)
    print("Tab:\tafter tab\nNew line")


creating_demo()

Python 6
----------
Tab:	after tab
New line


## Lesson 2: Indexing and slicing

*Level: Basics*

Each character has a position. Python counts from 0 at the left, and from −1 at the right:



```text
 P   y   t   h   o   n
 0   1   2   3   4   5
-6  -5  -4  -3  -2  -1
```



`s[i]` gets one character. `s[start:stop:step]` gets a **slice** — stop is not included, and any part can be left out:



| Slice | Result for "Python" |
| --- | --- |
| `s[0:2]` | 'Py' |
| `s[2:]` | 'thon' |
| `s[:-1]` | 'Pytho' |
| `s[::2]` | 'Pto' |
| `s[::-1]` | 'nohtyP' |



`s[10]` raises IndexError, but slices never do: `s[2:100]` just stops at the end.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def slicing_demo():
    s = "Python"
    first = s[0]
    last = s[-1]
    middle = s[1:4]
    every_second = s[::2]
    backwards = s[::-1]
    print(first, last, middle, every_second, backwards)


slicing_demo()

P n yth Pto nohtyP


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 3: Strings are immutable

*Level: Core*

You can't change a character inside a string: `s[0] = "J"` raises `TypeError`. Every "change" creates a **new** string:



```python
s = "Python"
s = "J" + s[1:]       # a new string "Jython"; the name s now points to it
t = s.upper()         # s is unchanged; t is a new string
```



Because of this, building a string with `+=` in a long loop copies it again and again. Collect pieces in a list and join once at the end:



```python
parts = []
for x in data:
    parts.append(str(x))
result = ",".join(parts)
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def immutable_demo():
    s = "Python"
    t = s.upper()
    s = "J" + s[1:]
    parts = []
    for n in [1, 2, 3]:
        parts.append(str(n))
    joined = "-".join(parts)
    print(s, t, joined)


immutable_demo()

Jython PYTHON 1-2-3


**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q8](#q8), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 4: The most useful string methods

*Level: Core*

| Method | Does | Example → result |
| --- | --- | --- |
| `lower()` / `upper()` | change case | `"Hi".upper()` → 'HI' |
| `title()` / `capitalize()` | capital first letters | `"a b".title()` → 'A B' |
| `strip()` | remove spaces at both ends | `"  x ".strip()` → 'x' |
| `split(sep)` | cut into a list | `"a,b".split(",")` → ['a', 'b'] |
| `sep.join(list)` | glue a list together | `"-".join(["a","b"])` → 'a-b' |
| `replace(old, new)` | replace all | `"aa".replace("a","b")` → 'bb' |
| `find(sub)` | first index, or −1 | `"hello".find("l")` → 2 |
| `count(sub)` | occurrences | `"hello".count("l")` → 2 |
| `startswith` / `endswith` | prefix / suffix test | `"a.py".endswith(".py")` → True |
| `isdigit`, `isalpha`, `isalnum`, `isspace` | what kind of characters | `"42".isdigit()` → True |

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def methods_demo():
    raw = "  Learn Python, Have Fun  "
    clean = raw.strip()
    lower = clean.lower()
    words = lower.split(", ")
    joined = " | ".join(words)
    where = clean.find("Python")
    print(clean)
    print(words, joined, where)


methods_demo()

Learn Python, Have Fun
['learn python', 'have fun'] learn python | have fun 6


**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q5](#q5), [Q3](#q3), [Q9](#q9), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 5: Working character by character

*Level: Core*

Each character has a number, its **Unicode code point**: `ord("a")` is 97, and `chr(97)` is "a". Letters are consecutive, so arithmetic on codes lets you shift letters (Caesar cipher) or find a letter's position (`ord(c) − ord("a")`).



Characters compare by their codes, so `"a" < "b"` and `"Z" < "a"` (all capitals come before lower case).



Counting characters is a very common task — a dict (or `collections.Counter`) of character → count solves anagrams, most frequent and first unique character.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def char_codes_demo():
    word = "abc"
    codes = [ord(c) for c in word]
    shifted = "".join(chr(code + 1) for code in codes)
    counts = {}
    for ch in "banana":
        counts[ch] = counts.get(ch, 0) + 1
    print(codes, shifted, counts)


char_codes_demo()

[97, 98, 99] bcd {'b': 1, 'a': 3, 'n': 2}


**✏️ Your turn:** you now have what you need for [Q18](#q18), [Q14](#q14), [Q15](#q15), [Q19](#q19), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 6: Formatting text

*Level: Advanced*

f-strings support a **format specification** after a colon: `{value:fill align width .precision type}`.



| Spec | Meaning | Result |
| --- | --- | --- |
| `f"{3.14159:.2f}"` | 2 decimals | 3.14 |
| `f"{42:5}"` | width 5 (numbers align right) | " 42" |
| `f"{'hi':*^8}"` | centre in 8, fill with \* | \*\*\*hi\*\*\* |
| `f"{1234567:,}"` | thousands separator | 1,234,567 |
| `f"{0.075:.1%}"` | percentage | 7.5% |
| `f"{255:b} {255:x}"` | binary / hex | 11111111 ff |
| `f"{x=}"` | debug: name and value | x=5 |



Grouping repeated characters (run-length encoding) and validating text are classic exercises that combine loops, conditions and methods.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def format_demo():
    items = [("Pen", 1.5), ("Notebook", 12.25)]
    for name, price in items:
        print(f"{name:<10}|{price:>8.2f}")
    total = sum(p for _, p in items)
    print(f"{'TOTAL':*^18}")
    print(f"{total=}", f"{0.075:.1%}")


format_demo()

Pen       |    1.50
Notebook  |   12.25
******TOTAL*******
total=13.75 7.5%


**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q16](#q16), [Q20](#q20), [Q10](#q10). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Reverse a String

🟢 Warm-up · *Slicing*

Return the string reversed.

**Examples**

- `s = 'hello'` → `'olleh'`
- `s = 'a'` → `'a'`
- `s = ''` → `''`

In [ ]:
def reverse_string(s):
    """
    Return the string reversed.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse_string, [
    (('hello',), 'olleh'),
    (('a',), 'a'),
    (('',), ''),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a new string by adding characters from the end to the start.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slice with step −1: `s[::-1]`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Build a new string by adding characters from the end to the start.

```python
def reverse_string(s):
    result = ""
    for i in range(len(s) - 1, -1, -1):
        result = result + s[i]
    return result
```

**Cost:** O(n²) — each + copies time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Slice with step −1: `s[::-1]`.

```python
def reverse_string(s):
    return s[::-1]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a new string by adding characters from the end to the start. That costs O(n²) — each + copies time and O(n) space."

"I can do better: Slice with step −1: `s[::-1]`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/reverse-a-string)

<a id="q2"></a>
### Q2. Palindrome Word

🟢 Warm-up · *Comparison · lower()*

Return True if the word reads the same backwards (ignore letter case).

**Examples**

- `word = 'Level'` → `True`
- `word = 'hello'` → `False`
- `word = 'A'` → `True`

In [ ]:
def is_palindrome(word):
    """
    Return True if the word reads the same backwards (ignore letter case).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_palindrome, [
    (('Level',), True),
    (('hello',), False),
    (('A',), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare characters from both ends moving inward.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Lower-case it and compare with its reverse.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compare characters from both ends moving inward.

```python
def is_palindrome(word):
    word = word.lower()
    i, j = 0, len(word) - 1
    while i < j:
        if word[i] != word[j]:
            return False
        i += 1
        j -= 1
    return True
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Lower-case it and compare with its reverse.

```python
def is_palindrome(word):
    w = word.lower()
    return w == w[::-1]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare characters from both ends moving inward. That costs O(n) time and O(1) space."

"I can do better: Lower-case it and compare with its reverse. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/palindrome-word)

<a id="q3"></a>
### Q3. Count a Character

🟢 Warm-up · *count()*

Return how many times character `ch` appears in `s`.

**Examples**

- `s = 'banana', ch = 'a'` → `3`
- `s = 'hello', ch = 'z'` → `0`
- `s = 'aaa', ch = 'a'` → `3`

In [ ]:
def count_char(s, ch):
    """
    Return how many times character ch appears in s.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_char, [
    (('banana', 'a'), 3),
    (('hello', 'z'), 0),
    (('aaa', 'a'), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and compare each character.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`s.count(ch)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop and compare each character.

```python
def count_char(s, ch):
    count = 0
    for c in s:
        if c == ch:
            count += 1
    return count
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `s.count(ch)`.

```python
def count_char(s, ch):
    return s.count(ch)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and compare each character. That costs O(n) time and O(1) space."

"I can do better: `s.count(ch)`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/count-a-character)

<a id="q4"></a>
### Q4. Capitalise Each Word

🟢 Warm-up · *split · join · capitalize*

Return the sentence with the first letter of every word in upper case and the rest lower case. Words are separated by single spaces.

**Examples**

- `sentence = 'hello world'` → `'Hello World'`
- `sentence = 'pYTHON is FUN'` → `'Python Is Fun'`

In [ ]:
def capitalise_words(sentence):
    """
    Return the sentence with the first letter of every word in upper case and
    the rest lower case. Words are separated by single spaces.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(capitalise_words, [
    (('hello world',), 'Hello World'),
    (('pYTHON is FUN',), 'Python Is Fun'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the characters, upper-casing the one after a space.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Split into words, capitalise each, join with spaces.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Walk the characters, upper-casing the one after a space.

```python
def capitalise_words(sentence):
    result = ""
    new_word = True
    for ch in sentence:
        if ch == " ":
            result += ch
            new_word = True
        elif new_word:
            result += ch.upper()
            new_word = False
        else:
            result += ch.lower()
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Split into words, capitalise each, join with spaces.

```python
def capitalise_words(sentence):
    return " ".join(word.capitalize() for word in sentence.split(" "))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the characters, upper-casing the one after a space. That costs O(n) time and O(n) space."

"I can do better: Split into words, capitalise each, join with spaces. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/capitalise-each-word)

<a id="q5"></a>
### Q5. Count Words

🟢 Warm-up · *split()*

Return the number of words in the text (words are separated by any amount of whitespace).

**Examples**

- `text = 'The quick  brown fox'` → `4`
- `text = '   '` → `0`
- `text = 'one'` → `1`

In [ ]:
def count_words(text):
    """
    Return the number of words in the text (words are separated by any amount of
    whitespace).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_words, [
    (('The quick  brown fox',), 4),
    (('   ',), 0),
    (('one',), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the characters and count every place where a word starts.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`split()` with no argument splits on runs of whitespace and ignores the ends.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Walk the characters and count every place where a word starts.

```python
def count_words(text):
    count = 0
    in_word = False
    for ch in text:
        if ch.isspace():
            in_word = False
        elif not in_word:
            count += 1
            in_word = True
    return count
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `split()` with no argument splits on runs of whitespace and ignores the ends.

```python
def count_words(text):
    return len(text.split())
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the characters and count every place where a word starts. That costs O(n) time and O(1) space."

"I can do better: `split()` with no argument splits on runs of whitespace and ignores the ends. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/count-words)

<a id="q6"></a>
### Q6. Remove Vowels

🟢 Warm-up · *Building strings*

Return the text with all vowels (a, e, i, o, u in either case) removed.

**Examples**

- `text = 'Hello World'` → `'Hll Wrld'`
- `text = 'aeiou'` → `''`
- `text = 'xyz'` → `'xyz'`

In [ ]:
def remove_vowels(text):
    """
    Return the text with all vowels (a, e, i, o, u in either case) removed.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(remove_vowels, [
    (('Hello World',), 'Hll Wrld'),
    (('aeiou',), ''),
    (('xyz',), 'xyz'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add each non-vowel to a result string with +=.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Join the kept characters from a generator — one allocation.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Add each non-vowel to a result string with +=.

```python
def remove_vowels(text):
    result = ""
    for ch in text:
        if ch.lower() not in "aeiou":
            result += ch
    return result
```

**Cost:** O(n²) worst time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Join the kept characters from a generator — one allocation.

```python
def remove_vowels(text):
    return "".join(ch for ch in text if ch.lower() not in "aeiou")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add each non-vowel to a result string with +=. That costs O(n²) worst time and O(n) space."

"I can do better: Join the kept characters from a generator — one allocation. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/remove-vowels)

<a id="q7"></a>
### Q7. Title Initials

🟢 Warm-up · *split · indexing*

Return the initials of a full name in upper case, e.g. 'ada king lovelace' → 'AKL'.

**Examples**

- `name = 'ada king lovelace'` → `'AKL'`
- `name = 'Grace Hopper'` → `'GH'`

In [ ]:
def initials(name):
    """
    Return the initials of a full name in upper case, e.g. 'ada king lovelace' →
    'AKL'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(initials, [
    (('ada king lovelace',), 'AKL'),
    (('Grace Hopper',), 'GH'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk characters, taking the first one and every character after a space.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Split into words and join their first letters.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Walk characters, taking the first one and every character after a space.

```python
def initials(name):
    result = name[0].upper()
    for i in range(1, len(name)):
        if name[i - 1] == " ":
            result += name[i].upper()
    return result
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Split into words and join their first letters.

```python
def initials(name):
    return "".join(word[0].upper() for word in name.split())
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk characters, taking the first one and every character after a space. That costs O(n) time and O(k) space."

"I can do better: Split into words and join their first letters. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/title-initials)

<a id="q8"></a>
### Q8. Replace Spaces

🟢 Warm-up · *replace()*

Replace every space in `text` with `'_'`.

**Examples**

- `text = 'snake case me'` → `'snake_case_me'`
- `text = 'nospace'` → `'nospace'`

In [ ]:
def underscores(text):
    """
    Replace every space in text with '_'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(underscores, [
    (('snake case me',), 'snake_case_me'),
    (('nospace',), 'nospace'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a new string character by character.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`text.replace(' ', '_')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Build a new string character by character.

```python
def underscores(text):
    out = []
    for ch in text:
        out.append("_" if ch == " " else ch)
    return "".join(out)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `text.replace(' ', '_')`.

```python
def underscores(text):
    return text.replace(" ", "_")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a new string character by character. That costs O(n) time and O(n) space."

"I can do better: `text.replace(' ', '_')`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/replace-spaces)

<a id="q9"></a>
### Q9. Swap Case

🟢 Warm-up · *Character methods*

Return the text with upper-case letters made lower-case and vice versa.

**Examples**

- `text = 'Hello World'` → `'hELLO wORLD'`
- `text = '123abc'` → `'123ABC'`

In [ ]:
def swap_case(text):
    """
    Return the text with upper-case letters made lower-case and vice versa.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(swap_case, [
    (('Hello World',), 'hELLO wORLD'),
    (('123abc',), '123ABC'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each character and convert it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`str.swapcase()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check each character and convert it.

```python
def swap_case(text):
    out = ""
    for ch in text:
        if ch.isupper():
            out += ch.lower()
        elif ch.islower():
            out += ch.upper()
        else:
            out += ch
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `str.swapcase()`.

```python
def swap_case(text):
    return text.swapcase()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each character and convert it. That costs O(n) time and O(n) space."

"I can do better: `str.swapcase()`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/swap-case)

<a id="q10"></a>
### Q10. Longest Word

🟢 Warm-up · *max with key*

Return the longest word in a sentence (the first one if there's a tie).

**Examples**

- `sentence = 'I love programming in Python'` → `'programming'`
- `sentence = 'a bb cc'` → `'bb'`

In [ ]:
def longest_word(sentence):
    """
    Return the longest word in a sentence (the first one if there's a tie).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_word, [
    (('I love programming in Python',), 'programming'),
    (('a bb cc',), 'bb'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the longest word while looping over the split words.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max(words, key=len)` returns the first longest.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Track the longest word while looping over the split words.

```python
def longest_word(sentence):
    best = ""
    for word in sentence.split():
        if len(word) > len(best):
            best = word
    return best
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `max(words, key=len)` returns the first longest.

```python
def longest_word(sentence):
    return max(sentence.split(), key=len)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the longest word while looping over the split words. That costs O(n) time and O(n) space."

"I can do better: `max(words, key=len)` returns the first longest. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/longest-word)

<a id="q11"></a>
### Q11. Mask a Card Number

🟢 Warm-up · *Slicing · repetition*

Replace every character except the last 4 with '#'.

**Examples**

- `card = '4556364607935616'` → `'############5616'`
- `card = '1234'` → `'1234'`
- `card = '12'` → `'12'`

In [ ]:
def mask(card):
    """
    Replace every character except the last 4 with '#'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(mask, [
    (('4556364607935616',), '############5616'),
    (('1234',), '1234'),
    (('12',), '12'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with an index and decide character by character.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Repeat '#' for the hidden length and add the last 4 with a slice.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop with an index and decide character by character.

```python
def mask(card):
    out = ""
    for i in range(len(card)):
        if i < len(card) - 4:
            out += "#"
        else:
            out += card[i]
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Repeat '#' for the hidden length and add the last 4 with a slice.

```python
def mask(card):
    return "#" * max(len(card) - 4, 0) + card[-4:]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with an index and decide character by character. That costs O(n) time and O(n) space."

"I can do better: Repeat '#' for the hidden length and add the last 4 with a slice. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/mask-a-card-number)

<a id="q12"></a>
### Q12. Only Digits

🟢 Warm-up · *Filtering characters*

Return just the digits from a string, e.g. '+91 98-765' → '9198765'.

**Examples**

- `s = '+91 98-765'` → `'9198765'`
- `s = 'abc'` → `''`
- `s = '2024'` → `'2024'`

In [ ]:
def only_digits(s):
    """
    Return just the digits from a string, e.g. '+91 98-765' → '9198765'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(only_digits, [
    (('+91 98-765',), '9198765'),
    (('abc',), ''),
    (('2024',), '2024'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare each character with '0'…'9'.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Join the characters for which `isdigit()` is True.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compare each character with '0'…'9'.

```python
def only_digits(s):
    out = ""
    for ch in s:
        if ch >= "0" and ch <= "9":
            out += ch
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Join the characters for which `isdigit()` is True.

```python
def only_digits(s):
    return "".join(ch for ch in s if ch.isdigit())
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare each character with '0'…'9'. That costs O(n) time and O(n) space."

"I can do better: Join the characters for which `isdigit()` is True. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/only-digits)

<a id="q13"></a>
### Q13. Centre a Title

🟢 Warm-up · *Format specifiers*

Return `title` centred in a field of `width` characters, padded with '\*'. Extra padding goes on the right.

**Examples**

- `title = 'Hi', width = 6` → `'**Hi**'`
- `title = 'Py', width = 5` → `'*Py**'`
- `title = 'Long', width = 2` → `'Long'`

In [ ]:
def centre(title, width):
    """
    Return title centred in a field of width characters, padded with '*'. Extra
    padding goes on the right.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(centre, [
    (('Hi', 6), '**Hi**'),
    (('Py', 5), '*Py**'),
    (('Long', 2), 'Long'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Work out left and right padding with arithmetic and build the string.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`title.center(width, '*')` (or f-string `:*^{width}`).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Work out left and right padding with arithmetic and build the string.

```python
def centre(title, width):
    total = max(width - len(title), 0)
    left = total // 2
    right = total - left
    return "*" * left + title + "*" * right
```

**Cost:** O(width) time, O(width) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `title.center(width, '*')` (or f-string `:*^{width}`).

```python
def centre(title, width):
    return f"{title:*^{width}}"
```

**Cost:** O(width) time, O(width) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Work out left and right padding with arithmetic and build the string. That costs O(width) time and O(width) space."

"I can do better: `title.center(width, '*')` (or f-string `:*^{width}`). That's O(width) time and O(width) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/centre-a-title)

<a id="q14"></a>
### Q14. Anagram Check

🟡 Interview standard · *sorted · Counter*

Return True if two words use exactly the same letters (ignore case).

**Examples**

- `a = 'Listen', b = 'Silent'` → `True`
- `a = 'abc', b = 'abd'` → `False`
- `a = 'aab', b = 'abb'` → `False`

In [ ]:
def is_anagram(a, b):
    """
    Return True if two words use exactly the same letters (ignore case).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_anagram, [
    (('Listen', 'Silent'), True),
    (('abc', 'abd'), False),
    (('aab', 'abb'), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each letter of the first word, find and remove it from a list of the second word's letters.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compare the sorted letters (or `Counter`s).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each letter of the first word, find and remove it from a list of the second word's letters.

```python
def is_anagram(a, b):
    letters = list(b.lower())
    for ch in a.lower():
        if ch in letters:
            letters.remove(ch)
        else:
            return False
    return len(letters) == 0
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Compare the sorted letters (or `Counter`s).

```python
def is_anagram(a, b):
    return sorted(a.lower()) == sorted(b.lower())
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each letter of the first word, find and remove it from a list of the second word's letters. That costs O(n²) time and O(n) space."

"I can do better: Compare the sorted letters (or `Counter`s). That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/anagram-check)

<a id="q15"></a>
### Q15. Most Frequent Character

🟡 Interview standard · *Counting*

Return the character that appears most often in `s` (ties → the one that appears first).

**Examples**

- `s = 'banana'` → `'a'`
- `s = 'abcab'` → `'a'`
- `s = 'zzyy'` → `'z'`

In [ ]:
def most_frequent(s):
    """
    Return the character that appears most often in s (ties → the one that
    appears first).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(most_frequent, [
    (('banana',), 'a'),
    (('abcab',), 'a'),
    (('zzyy',), 'z'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each character, count it with a loop over the whole string.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count once with a dict, then pick the max by count; dicts keep insertion order so ties favour the earliest.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each character, count it with a loop over the whole string.

```python
def most_frequent(s):
    best, best_count = "", 0
    for ch in s:
        c = 0
        for other in s:
            if other == ch:
                c += 1
        if c > best_count:
            best, best_count = ch, c
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Count once with a dict, then pick the max by count; dicts keep insertion order so ties favour the earliest.

```python
def most_frequent(s):
    counts = {}
    for ch in s:
        counts[ch] = counts.get(ch, 0) + 1
    return max(counts, key=counts.get)
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each character, count it with a loop over the whole string. That costs O(n²) time and O(1) space."

"I can do better: Count once with a dict, then pick the max by count; dicts keep insertion order so ties favour the earliest. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/most-frequent-character)

<a id="q16"></a>
### Q16. Compress Repeated Letters

🟡 Interview standard · *Run-length encoding*

Compress runs of the same letter: 'aaabccdddd' → 'a3b1c2d4'.

**Examples**

- `s = 'aaabccdddd'` → `'a3b1c2d4'`
- `s = 'abc'` → `'a1b1c1'`
- `s = ''` → `''`

In [ ]:
def compress(s):
    """
    Compress runs of the same letter: 'aaabccdddd' → 'a3b1c2d4'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compress, [
    (('aaabccdddd',), 'a3b1c2d4'),
    (('abc',), 'a1b1c1'),
    (('',), ''),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk with an index and an inner while loop that measures each run.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.groupby` groups consecutive equal letters for you.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Walk with an index and an inner while loop that measures each run.

```python
def compress(s):
    out = ""
    i = 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out += s[i] + str(j - i)
        i = j
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `itertools.groupby` groups consecutive equal letters for you.

```python
def compress(s):
    from itertools import groupby
    return "".join(f"{ch}{len(list(run))}" for ch, run in groupby(s))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk with an index and an inner while loop that measures each run. That costs O(n) time and O(n) space."

"I can do better: `itertools.groupby` groups consecutive equal letters for you. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/compress-repeated-letters)

<a id="q17"></a>
### Q17. Is It a Pangram?

🟡 Interview standard · *Sets of characters*

Return True if the sentence contains every letter a–z at least once.

**Examples**

- `sentence = 'The quick brown fox jumps over the lazy dog'` → `True`
- `sentence = 'Hello world'` → `False`

In [ ]:
def is_pangram(sentence):
    """
    Return True if the sentence contains every letter a–z at least once.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_pangram, [
    (('The quick brown fox jumps over the lazy dog',), True),
    (('Hello world',), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each of the 26 letters, check whether it's in the sentence.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Turn the lower-cased sentence into a set and check the alphabet is a subset of it.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each of the 26 letters, check whether it's in the sentence.

```python
def is_pangram(sentence):
    s = sentence.lower()
    for code in range(ord("a"), ord("z") + 1):
        if chr(code) not in s:
            return False
    return True
```

**Cost:** O(26 · n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Turn the lower-cased sentence into a set and check the alphabet is a subset of it.

```python
def is_pangram(sentence):
    import string
    return set(string.ascii_lowercase) <= set(sentence.lower())
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each of the 26 letters, check whether it's in the sentence. That costs O(26 · n) time and O(1) space."

"I can do better: Turn the lower-cased sentence into a set and check the alphabet is a subset of it. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/is-it-a-pangram)

<a id="q18"></a>
### Q18. Caesar Cipher

🟡 Interview standard · *ord · chr · %*

Shift every lowercase letter forward by `k` places in the alphabet (wrapping z → a). Leave other characters as they are.

**Examples**

- `text = 'abc xyz', k = 3` → `'def abc'`
- `text = 'hello', k = 13` → `'uryyb'`
- `text = 'a-b', k = 1` → `'b-c'`

In [ ]:
def caesar(text, k):
    """
    Shift every lowercase letter forward by k places in the alphabet (wrapping z
    → a). Leave other characters as they are.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(caesar, [
    (('abc xyz', 3), 'def abc'),
    (('hello', 13), 'uryyb'),
    (('a-b', 1), 'b-c'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Look up each letter's position in the alphabet string with `index`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use character codes: `chr((ord(c) − 97 + k) % 26 + 97)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Look up each letter's position in the alphabet string with `index`.

```python
def caesar(text, k):
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    out = ""
    for ch in text:
        if ch in alphabet:
            pos = alphabet.index(ch)
            out += alphabet[(pos + k) % 26]
        else:
            out += ch
    return out
```

**Cost:** O(26 · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use character codes: `chr((ord(c) − 97 + k) % 26 + 97)`.

```python
def caesar(text, k):
    return "".join(chr((ord(c) - 97 + k) % 26 + 97) if "a" <= c <= "z" else c for c in text)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Look up each letter's position in the alphabet string with `index`. That costs O(26 · n) time and O(n) space."

"I can do better: Use character codes: `chr((ord(c) − 97 + k) % 26 + 97)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/caesar-cipher)

<a id="q19"></a>
### Q19. First Non-Repeating Character

🟡 Interview standard · *Counting · order*

Return the first character that appears exactly once, or '' if none.

**Examples**

- `s = 'swiss'` → `'w'`
- `s = 'aabb'` → `''`
- `s = 'python'` → `'p'`

In [ ]:
def first_unique(s):
    """
    Return the first character that appears exactly once, or '' if none.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(first_unique, [
    (('swiss',), 'w'),
    (('aabb',), ''),
    (('python',), 'p'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each character, count it by scanning the whole string.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count all characters once with a dict, then scan again for the first count of 1.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each character, count it by scanning the whole string.

```python
def first_unique(s):
    for ch in s:
        if s.count(ch) == 1:
            return ch
    return ""
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Count all characters once with a dict, then scan again for the first count of 1.

```python
def first_unique(s):
    from collections import Counter
    counts = Counter(s)
    return next((ch for ch in s if counts[ch] == 1), "")
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each character, count it by scanning the whole string. That costs O(n²) time and O(1) space."

"I can do better: Count all characters once with a dict, then scan again for the first count of 1. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/first-non-repeating-character)

<a id="q20"></a>
### Q20. Valid Username

🟡 Interview standard · *Several checks*

A valid username is 3–12 characters, starts with a letter, and contains only letters, digits or '\_'. Return True or False.

**Examples**

- `name = 'ada_99'` → `True`
- `name = '9lives'` → `False`
- `name = 'ab'` → `False`

In [ ]:
def valid_username(name):
    """
    A valid username is 3–12 characters, starts with a letter, and contains only
    letters, digits or '_'. Return True or False.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(valid_username, [
    (('ada_99',), True),
    (('9lives',), False),
    (('ab',), False),
    (('hello world',), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check length, then the first character, then loop over the rest with explicit comparisons.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Combine `isalpha`, a length range check and `all()` over the characters.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check length, then the first character, then loop over the rest with explicit comparisons.

```python
def valid_username(name):
    if len(name) < 3 or len(name) > 12:
        return False
    first = name[0]
    if not (("a" <= first <= "z") or ("A" <= first <= "Z")):
        return False
    for ch in name:
        ok = ("a" <= ch <= "z") or ("A" <= ch <= "Z") or ("0" <= ch <= "9") or ch == "_"
        if not ok:
            return False
    return True
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Combine `isalpha`, a length range check and `all()` over the characters.

```python
def valid_username(name):
    return 3 <= len(name) <= 12 and name[0].isalpha() and all(c.isalnum() or c == "_" for c in name)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check length, then the first character, then loop over the rest with explicit comparisons. That costs O(n) time and O(1) space."

"I can do better: Combine `isalpha`, a length range check and `all()` over the characters. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/strings/valid-username)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why are strings immutable in Python?</b></summary>

So their hash value never changes (making them usable as dict keys and set items), so they can be shared safely, and so Python can optimise memory by reusing them.

</details>

<details>
<summary><b>2. find() vs index()?</b></summary>

Both return the first position of a substring. `find` returns −1 if it's missing; `index` raises `ValueError`.

</details>

<details>
<summary><b>3. How do you reverse a string?</b></summary>

`s[::-1]`, or `''.join(reversed(s))`.

</details>

<details>
<summary><b>4. Why use join instead of + in a loop?</b></summary>

Each `+` creates a new string and copies the old content, so building a string of n pieces costs O(n²). `''.join(list)` computes the final size once and copies each piece once — O(n).

</details>

<details>
<summary><b>5. What is the difference between split() and split(' ')?</b></summary>

`split()` splits on any run of whitespace and drops empty pieces at the ends. `split(' ')` splits on every single space, so double spaces produce empty strings.

</details>

<details>
<summary><b>6. What does s[::-1] mean?</b></summary>

A slice from the end to the start with step −1 — every character in reverse order.

</details>

<details>
<summary><b>7. isdigit vs isnumeric vs isdecimal?</b></summary>

All test for number characters. `isdecimal` is strictest (0–9 and other decimal digits), `isdigit` also accepts superscripts like ², and `isnumeric` also accepts fractions and numerals like ½ or Ⅻ. For simple input checks `isdigit` is usual.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Creating strings
- [ ] I can explain: Indexing and slicing
- [ ] I can explain: Strings are immutable
- [ ] I can explain: The most useful string methods
- [ ] I can explain: Working character by character
- [ ] I can explain: Formatting text

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Compress Repeated Letters
- [ ] [Q17](#q17) Is It a Pangram?
- [ ] [Q18](#q18) Caesar Cipher
- [ ] [Q19](#q19) First Non-Repeating Character
- [ ] [Q20](#q20) Valid Username

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.